# Detecting Motor Anomalies

Preventing motor anomalies is a bit more complicated than battery issues. Usually, motors operate in a certain range of power, but sometimes they may present anomalous behavior. Their power consumption can go to high, due to environmental issues, or too low, due to aging issues.

As usual, let's start by recovering and looking at data:

In [ ]:
%store -r data
data.head()

In [ ]:
%store -r bucket
bucket

# Exploratory Data Analysis

In [ ]:
train_data = data[["motor_peak_mA"]]
train_data = train_data[train_data["motor_peak_mA"] > 0]
train_data.head()

In [ ]:
train_data.describe()

In [ ]:
import matplotlib.pyplot as plt
train_data.plot(rot=30)

## Synthetic Ground Truth

In [ ]:
anomalies = data[["motor_peak_mA"]]
anomalies = anomalies[anomalies["motor_peak_mA"] > 0]
anomalies.info()

In [ ]:
from sklearn.model_selection import train_test_split
train_data, test_dataframe = train_test_split(anomalies, test_size=0.2)

In [ ]:
test_data = test_dataframe.copy()
test_data["anomaly"] = test_data["motor_peak_mA"] > 4000
test_data["anomaly"] = test_data["anomaly"] | (test_data["motor_peak_mA"] > 50) & (test_data["motor_peak_mA"] < 200)
test_data["anomaly"] = test_data["anomaly"].astype(int)
test_data.groupby("anomaly").count().head()

In [ ]:
test_data.describe()

In [ ]:
train_data.describe()

# Random Cut Forest Training

In [ ]:
train_array  = train_data.values
test_array   = test_data[["motor_peak_mA"]].values
labels_array = test_data["anomaly"].values

In [ ]:
import io, struct, boto3

s3bucket = boto3.resource('s3').Bucket(bucket)

def _varint(n):
    buf = b''
    while n > 0x7f:
        buf += bytes([0x80 | (n & 0x7f)])
        n >>= 7
    return buf + bytes([n])

def _map_entry(field_num, key, floats):
    packed = struct.pack('{}f'.format(len(floats)), *floats)
    tensor = b'\x0a' + _varint(len(packed)) + packed
    value  = b'\x12' + _varint(len(tensor)) + tensor
    kv     = b'\x0a' + _varint(len(key)) + key.encode() + b'\x12' + _varint(len(value)) + value
    return _varint((field_num << 3) | 2) + _varint(len(kv)) + kv

def upload_recordio(array, key, labels=None):
    buf = io.BytesIO()
    for i, row in enumerate(array.astype('float32')):
        rec = _map_entry(1, 'values', row.tolist())
        if labels is not None:
            rec += _map_entry(2, 'values', [float(labels[i])])
        buf.write(struct.pack('II', 0xced7230a, len(rec)))
        buf.write(rec)
        buf.write(b'\x00' * ((4 - len(rec) % 4) % 4))
    buf.seek(0)
    s3bucket.Object(key).upload_fileobj(buf)

In [ ]:
import time
from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core import image_uris

sagemaker_session = Session()
region = sagemaker_session.boto_region_name
role = get_execution_role()
rcf_container = image_uris.retrieve(framework='randomcutforest', region=region)

prefix    = "mt-motor-anomaly"
train_key = "{}/input/train.rio".format(prefix)
test_key  = "{}/input/test.rio".format(prefix)

upload_recordio(train_array, train_key)
upload_recordio(test_array, test_key, labels_array)
print(rcf_container)

In [ ]:
from sagemaker.core.resources import TrainingJob
from sagemaker.core.shapes import (
    AlgorithmSpecification, Channel, DataSource, S3DataSource,
    ResourceConfig, StoppingCondition, OutputDataConfig
)

job_name = "mt-motor-anomaly-" + time.strftime("%Y-%m-%d-%H-%M-%S", time.gmtime())

training_job = TrainingJob.create(
    training_job_name=job_name,
    algorithm_specification=AlgorithmSpecification(
        training_image=rcf_container,
        training_input_mode='File'
    ),
    role_arn=role,
    input_data_config=[
        Channel(
            channel_name='train',
            content_type='application/x-recordio-protobuf',
            data_source=DataSource(
                s3_data_source=S3DataSource(
                    s3_data_type='S3Prefix',
                    s3_uri='s3://{}/{}'.format(bucket, train_key),
                    s3_data_distribution_type='ShardedByS3Key'
                )
            )
        ),
        Channel(
            channel_name='test',
            content_type='application/x-recordio-protobuf',
            data_source=DataSource(
                s3_data_source=S3DataSource(
                    s3_data_type='S3Prefix',
                    s3_uri='s3://{}/{}'.format(bucket, test_key),
                    s3_data_distribution_type='FullyReplicated'
                )
            )
        ),
    ],
    output_data_config=OutputDataConfig(
        s3_output_path='s3://{}/{}/output'.format(bucket, prefix)
    ),
    resource_config=ResourceConfig(
        instance_type='ml.m5.large',
        instance_count=1,
        volume_size_in_gb=10
    ),
    hyper_parameters={
        'num_samples_per_tree': '512',
        'num_trees': '50',
        'feature_dim': '1',
        'eval_metrics': '["accuracy"]',
    },
    stopping_condition=StoppingCondition(max_runtime_in_seconds=3600)
)
training_job.wait()
print('Training job name:', job_name)

In [ ]:
model_artifacts = training_job.model_artifacts.s3_model_artifacts
print('Model artifacts:', model_artifacts)

In [ ]:
from sagemaker.core.resources import Model, EndpointConfig, Endpoint
from sagemaker.core.shapes import ContainerDefinition, ProductionVariant

model_name           = job_name + "-model"
endpoint_config_name = job_name + "-epc"
endpoint_name        = job_name + "-ep"

Model.create(
    model_name=model_name,
    primary_container=ContainerDefinition(
        image=rcf_container,
        model_data_url=model_artifacts
    ),
    execution_role_arn=role
)

EndpointConfig.create(
    endpoint_config_name=endpoint_config_name,
    production_variants=[
        ProductionVariant(
            variant_name='AllTraffic',
            model_name=model_name,
            instance_type='ml.m5.large',
            initial_instance_count=1
        )
    ]
)

rcf_endpoint = Endpoint.create(
    endpoint_name=endpoint_name,
    endpoint_config_name=endpoint_config_name
)
rcf_endpoint.wait_for_status(target_status='InService')
print('Endpoint:', endpoint_name)

In [ ]:
rcf_inference_endpoint = endpoint_name
%store rcf_inference_endpoint
rcf_inference_endpoint

In [ ]:
import json

sample_data = train_data[:5].values
csv_body = "\n".join(",".join(str(v) for v in row) for row in sample_data)

res = rcf_endpoint.invoke(body=csv_body, content_type="text/csv")
results = json.loads(res.body.read().decode('utf-8'))
results

In [ ]:
import pandas as pd
sigmas = 1

scores = [s["score"] for s in results["scores"]]
series = pd.Series(scores)
score_mean   = series.mean()
score_std    = series.std()
score_cutoff = score_mean + sigmas * score_std
(score_mean, series.max(), score_std, score_cutoff)

In [ ]:
detected = series[series > score_cutoff]
"{} anomalies detected".format(len(detected))

## Motor Maintenance

Now that we can detect anomalies in past data, let's combine that with forecasting for predictive [motor maintenance](mt-motor-maintenance.ipynb).